# Qwen2.5-7B LoRA (Kaggle Lite GPU)
Notebook nay duoc toi uu cho GPU nhe tren Kaggle (T4/P100/L4) voi QLoRA 4-bit.

In [ ]:
import os
import subprocess

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    print('HF_TOKEN detected')
else:
    print('HF_TOKEN not set. If model access is gated, training may fail.')

subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', '/kaggle/working/Qwen2.5B_finetune/requirements.txt'], check=True)

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/kaggle/working/Qwen2.5B_finetune')
assert PROJECT_DIR.exists(), f'Project directory not found: {PROJECT_DIR}'

# Example: /kaggle/input/<your-dataset>/train.csv
DATASET_CSV = Path('/kaggle/input/medical-qa/train.csv')
assert DATASET_CSV.exists(), f'Dataset CSV not found: {DATASET_CSV}'

os.chdir(PROJECT_DIR)
print('Working dir:', Path.cwd())
print('Dataset:', DATASET_CSV)

In [ ]:
import yaml
from pathlib import Path

template_cfg = Path('configs/train_qwen25_7b_kaggle_lite.yaml')
runtime_cfg = Path('/kaggle/working/train_qwen25_7b_kaggle_runtime.yaml')

with open(template_cfg, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['data']['csv_path'] = str(DATASET_CSV)

# Optional quick test on small subset:
# cfg['data']['limit_rows'] = 3000

with open(runtime_cfg, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=False)

print('Runtime config written to', runtime_cfg)
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
import os
import subprocess

env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['TOKENIZERS_PARALLELISM'] = 'false'

cmd = ['python', '-m', 'src.train_lora', '--config', '/kaggle/working/train_qwen25_7b_kaggle_runtime.yaml']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
from pathlib import Path
import subprocess

out_dir = Path('/kaggle/working/outputs/qwen25_7b_medqa_lora_kaggle')
log_file = out_dir / 'logs' / 'train.log'
metrics_file = out_dir / 'eval_metrics.yaml'

print('Output directory:', out_dir)
print('Log exists:', log_file.exists())
print('Metrics exists:', metrics_file.exists())

if log_file.exists():
    print('--- Last 60 log lines ---')
    print('\n'.join(log_file.read_text(encoding='utf-8').splitlines()[-60:]))

zip_path = Path('/kaggle/working/qwen25_7b_medqa_lora_kaggle.zip')
subprocess.run(['zip', '-r', str(zip_path), str(out_dir)], check=True)
print('Zipped artifacts:', zip_path)